In [23]:
import os 
import numpy as np
import pandas as pd

import anndata as ad
import scanpy as sc
import squidpy as sq
import gzip
import shutil


### Step 1: Load and prepare the xenium data:
1. Load the spatial transcript locations and cell metadata

In [26]:
input_file = "data/GSM8257566_7day2_transcripts.csv.gz"
output_file = "data/GSM8257566_7day2_transcripts.csv"

# Extract the file
with gzip.open(input_file, "rb") as f_in:
    with open(output_file, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print(f"File extracted: {output_file}")

File extracted: data/GSM8257566_7day2_transcripts.csv


In [31]:
input_file = "data/GSM8257566_7day2_cells.csv.gz"
output_file = "data/GSM8257566_7day2_cells.csv"

# Extract the file
with gzip.open(input_file, "rb") as f_in:
    with open(output_file, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print(f"File extracted: {output_file}")


File extracted: data/GSM8257566_7day2_cells.csv


In [36]:
transcript_file = "data/GSM8257566_7day2_transcripts.csv"
transcripts = pd.read_csv(transcript_file)

cells_file = "data/GSM8257566_7day2_cells.csv"
cells_df = pd.read_csv(cells_file)

In [37]:
# Count number of each gene per cell
gene_counts = transcripts.groupby(["cell_id", "feature_name"]).size().unstack(fill_value=0)

# Print shape: rows = cells, columns = genes
print(gene_counts.shape)  # (num_cells, num_genes)


(37759, 541)


In [39]:
gene_counts.head()

feature_name,2010300C02Rik,Acsbg1,Acta2,Acvrl1,Adamts2,Adamtsl1,Adgrl4,Aif1,Aldh1a2,Aldh1a3,...,Vwc2l,Vwf,Wfs1,Wnt5a,Zeb1,Zfp366,Zfp536,Zfpm2,eGFP,hPDGFB
cell_id,,,,,,,,,,,,,,,,,,,,,
UNASSIGNED,11518,20053,402,844,778,287,1079,1654,332,85,...,820,728,9013,479,2819,175,1435,392,1627,13
aaaaaeea-1,6,0,0,0,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0
aaacemln-1,10,0,0,0,0,0,0,0,0,0,...,0,0,3,0,0,0,2,0,0,0
aaafdicf-1,6,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,2,0,1,0
aaahlijd-1,5,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0


In [40]:
# Create AnnData object
adata = ad.AnnData(X=gene_counts.values)

# Assign cell and gene names
adata.obs_names = gene_counts.index  # Cell IDs
adata.var_names = gene_counts.columns  # Gene names

# Add spatial coordinates (X, Y)
cell_locations = transcripts.groupby("cell_id")[["x_location", "y_location"]].mean()
adata.obsm["spatial"] = cell_locations.loc[adata.obs_names].values

# Print summary
print(adata)


/var/folders/c8/b70xcn351z9fpn0fw0h0253r0000gn/T/ipykernel_67024/3693257087.py:2: FutureWarning: X.dtype being converted to np.float32 from int64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  adata = ad.AnnData(X=gene_counts.values)


AnnData object with n_obs × n_vars = 37759 × 541
    obsm: 'spatial'
